# Week 3 Exercise: Synthetic Data Generator

This tool creates synthetic datasets for business problems using direct data generation or code generation.

## Step 1: Set up your environment

Import required libraries and load API keys (Google Gemini and Anthropic Claude).

In [ ]:
import os
import re
import io
import sys
import json
import random
import datetime
import traceback
import subprocess
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import create_model
import anthropic
import gradio as gr
import pandas as pd

In [ ]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
claude = anthropic.Anthropic()

## Step 2: Add your local model

Connect to the local Ollama server using an OpenAI-compatible client.

In [ ]:
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## Step 3: Write your system prompt

Define a system message that enforces immediate code output without any conversational preamble or thinking notes.

In [ ]:
system_message = (
    "You are an assistant that creates synthetic datasets for business problems. "
    "You must start your response immediately with the raw dataset or code. "
    "Do not output internal thoughts, conversational text, greetings, or notes before or after."
)

## Step 4: Write your user prompt builder

Build a user prompt for either direct dataset generation or executable Python code generation.

In [ ]:
def build_user_prompt(business_problem, data_type, num_rows, file_format, generation_strategy="Code Generation", schema_json=None):
    schema_block = ""
    if schema_json:
        schema_block = f"\nEach record MUST conform to this JSON schema:\n{json.dumps(schema_json)}\n"

    is_direct = generation_strategy == "Direct Dataset"
    output_format = "JSON" if is_direct and file_format == "JSON" else file_format
    header = f"""
The business problem is: {business_problem}
Create a synthetic dataset of type: {data_type}
Number of rows: {num_rows}
Output format: {output_format}
{schema_block}
"""

    if is_direct and file_format == "JSON":
        return header + f"""CRITICAL RULES:
- Output a single JSON object of the form {{"rows": [...]}} where "rows" is an array of exactly {num_rows} records.
- Do not include any other keys, thoughts, notes, or explanation.
- Do not write Python code.
"""
    if is_direct:
        return header + f"""CRITICAL RULES:
- Output only the raw dataset in {file_format} format.
- Start immediately with the header row. Do not write any thoughts, notes, or explanation.
- Do not write Python code.
"""
    filename = f"synthetic_dataset.{file_format.lower()}"
    return header + f"""Write a clean, complete Python script using pandas and the standard library that builds this dataset and saves it to '{filename}'.
Wrap the script in a single ```python code block.
CRITICAL RULES:
- Start your reply IMMEDIATELY with ```python. DO NOT write any thoughts, intro, or greeting.
- End your reply with ```. DO NOT write any explanation after the code block.
- Include all necessary import statements at the top (e.g. import pandas as pd, random, datetime).
- Format all numbers with leading zeros (postal codes, phone numbers, IDs, dates) as strings (e.g. '01234'), never as integer literals.
- You MUST save the dataset with df.to_csv('{filename}', index=False) or the matching format writer.
- Print the saved file path at the end: print('File saved: {filename}')
"""

## Step 4b: Validate inputs and build an optional schema

Reject a bad row count or format before calling the model, instead of wasting an API call. Let the user optionally list field names and types, and turn that into a JSON schema the model must follow, instead of relying on prompt wording alone.

In [ ]:
MAX_ROWS = 5000


def validate_inputs(num_rows, file_format):
    if num_rows is None or num_rows <= 0:
        return "Number of rows must be a positive whole number."
    if num_rows > MAX_ROWS:
        return f"Number of rows must be {MAX_ROWS} or fewer."
    if file_format not in ("CSV", "JSON", "Markdown"):
        return f"Unsupported file format: {file_format}"
    return None


TYPE_MAP = {"str": str, "string": str, "int": int, "integer": int, "float": float, "bool": bool, "date": str}


def parse_schema_fields(schema_text):
    fields = {}
    if not schema_text or not schema_text.strip():
        return fields
    for part in schema_text.split(","):
        part = part.strip()
        if not part:
            continue
        name, _, type_hint = part.partition(":")
        name = name.strip()
        type_hint = type_hint.strip().lower() or "str"
        if name:
            fields[name] = TYPE_MAP.get(type_hint, str)
    return fields


def build_schema_json(schema_text):
    fields = parse_schema_fields(schema_text)
    if not fields:
        return None
    model = create_model("Row", **{name: (field_type, ...) for name, field_type in fields.items()})
    return model.model_json_schema()

## Step 5: Pick your generation strategy

You can choose between:
1. **Direct Dataset**: The model directly outputs rows of data (ideal for quick previews and small row counts).
2. **Code Generation**: The model outputs a Python script to build the dataset locally (scales to large datasets).

## Step 6: Call the model and stream the reply

Streaming functions for each provider with production model names hardcoded in the API calls.

In [ ]:
def generate_with_gemini(business_problem, data_type, num_rows, file_format, generation_strategy="Code Generation", schema_json=None):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": build_user_prompt(business_problem, data_type, num_rows, file_format, generation_strategy, schema_json)},
    ]
    kwargs = {}
    if generation_strategy == "Direct Dataset" and file_format == "JSON":
        kwargs["response_format"] = {"type": "json_object"}
    stream = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        stream=True,
        **kwargs
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


def generate_with_claude(business_problem, data_type, num_rows, file_format, generation_strategy="Code Generation", schema_json=None):
    prompt = build_user_prompt(business_problem, data_type, num_rows, file_format, generation_strategy, schema_json)
    result = ""
    with claude.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=4096,
        system=system_message,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text in stream.text_stream:
            result += text
            yield result


def generate_with_ollama(business_problem, data_type, num_rows, file_format, generation_strategy="Code Generation", schema_json=None):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": build_user_prompt(business_problem, data_type, num_rows, file_format, generation_strategy, schema_json)},
    ]
    kwargs = {}
    if generation_strategy == "Direct Dataset" and file_format == "JSON":
        kwargs["response_format"] = {"type": "json_object"}
    stream = ollama_client.chat.completions.create(
        model="gemma4:e4b-mlx",
        messages=messages,
        stream=True,
        **kwargs
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

## Step 7: Clean the streaming output

Remove thinking tags and markdown fences in real time so only pure code or clean data streams into the window.

In [ ]:
def remove_thought_tags(text):
    if "<thought>" in text:
        if "</thought>" in text:
            text = text.split("</thought>", 1)[1]
        else:
            return ""
    return text.lstrip()


def clean_stream(text, generation_strategy):
    text = remove_thought_tags(text)
    if not text:
        return ""

    if generation_strategy == "Direct Dataset":
        for fence in ["```csv", "```json", "```markdown", "```"]:
            if fence in text:
                text = text.split(fence, 1)[1]
                break
        if "```" in text:
            text = text.split("```", 1)[0]
        return text.strip()
    else:
        if "```python" in text:
            code_part = text.split("```python", 1)[1]
        elif "```" in text:
            code_part = text.split("```", 1)[1]
        else:
            stripped = text.strip()
            if stripped.startswith("import ") or stripped.startswith("from ") or stripped.startswith("#"):
                code_part = text
            else:
                return ""

        if "```" in code_part:
            code_part = code_part.split("```", 1)[0]

        return code_part.lstrip("\n")

## Step 7b: Batch large requests and repair malformed CSV rows

Split a Direct Dataset request over `BATCH_SIZE` rows into smaller calls, so the model does not truncate output on large requests. Also recover from malformed CSV rows with a fallback parser, instead of saving broken data as-is.

In [ ]:
BATCH_SIZE = 50


def strip_header_line(text, file_format):
    lines = text.splitlines()
    idx = 0
    while idx < len(lines) and not lines[idx].strip():
        idx += 1
    if idx >= len(lines):
        return ""
    if file_format == "Markdown" and idx + 1 < len(lines) and set(lines[idx + 1].strip()) <= set("-|: "):
        idx += 2
    else:
        idx += 1
    return "\n".join(lines[idx:])


def merge_batch(accumulated, batch_text, file_format, is_first_batch):
    if file_format == "JSON":
        try:
            existing_obj = json.loads(accumulated) if accumulated.strip() else {}
        except json.JSONDecodeError:
            existing_obj = {}
        existing_rows = existing_obj.get("rows", []) if isinstance(existing_obj, dict) else []

        try:
            batch_obj = json.loads(batch_text) if batch_text.strip() else {}
        except json.JSONDecodeError:
            batch_obj = {}
        if isinstance(batch_obj, dict):
            batch_rows = batch_obj.get("rows", [])
        elif isinstance(batch_obj, list):
            batch_rows = batch_obj
        else:
            batch_rows = []

        return json.dumps({"rows": existing_rows + batch_rows}, indent=2)

    cleaned = batch_text if is_first_batch else strip_header_line(batch_text, file_format)
    if not accumulated:
        return cleaned
    return accumulated + "\n" + cleaned


def generate_in_batches(provider, business_problem, data_type, num_rows, file_format, generation_strategy, schema_json):
    accumulated = ""
    remaining = num_rows
    batch_num = 0
    while remaining > 0:
        batch_rows = min(BATCH_SIZE, remaining)
        batch_num += 1
        batch_text = ""
        for full_response in provider(business_problem, data_type, batch_rows, file_format, generation_strategy, schema_json):
            batch_text = clean_stream(full_response, generation_strategy)
            yield merge_batch(accumulated, batch_text, file_format, batch_num == 1)
        accumulated = merge_batch(accumulated, batch_text, file_format, batch_num == 1)
        remaining -= batch_rows


def safe_parse_csv(text):
    try:
        return pd.read_csv(io.StringIO(text))
    except Exception:
        pass
    try:
        return pd.read_csv(io.StringIO(text), engine="python", on_bad_lines="skip")
    except Exception:
        pass
    lines = [line for line in text.splitlines() if line.strip()]
    if not lines:
        return pd.DataFrame()
    header = lines[0].split(",")
    rows = []
    for line in lines[1:]:
        fields = line.split(",")
        fields = (fields + [""] * len(header))[: len(header)]
        rows.append(fields)
    return pd.DataFrame(rows, columns=header)

## Step 7c: Run generated code in a sandboxed subprocess

Run the LLM-generated script in its own process, not inside this notebook's process. Give it a time limit. Do not give it the API keys. This stops a broken or malicious script from reading secrets, hanging forever, or corrupting notebook state.

In [ ]:
EXEC_TIMEOUT_SECONDS = 30
SAFE_ENV_KEYS = ("PATH", "HOME", "SYSTEMROOT", "TEMP", "TMP", "PYTHONPATH", "PYTHONHOME")


def build_safe_env():
    return {k: v for k, v in os.environ.items() if k in SAFE_ENV_KEYS}


def run_generated_code(clean_code, cwd):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as tmp:
        tmp.write(clean_code)
        script_path = tmp.name
    try:
        return subprocess.run(
            [sys.executable, script_path],
            cwd=cwd,
            env=build_safe_env(),
            capture_output=True,
            text=True,
            timeout=EXEC_TIMEOUT_SECONDS,
        )
    finally:
        os.remove(script_path)

## Step 8: Execute code or save dataset safely

Safely execute code with pre-loaded standard libraries or save clean dataset text to disk.

In [ ]:
def extract_pure_code(text):
    if not text:
        return ""
    text = remove_thought_tags(text)
    lines = text.strip().splitlines()
    clean_lines = [line for line in lines if not line.strip().startswith("```")]
    return "\n".join(clean_lines).strip()


def process_output(content, file_format, generation_strategy):
    if not content or not content.strip():
        return "No content to process. Please generate code or data first.", None

    cwd = os.path.abspath(".")
    ext = file_format.lower()
    if ext == "markdown":
        ext = "md"

    if generation_strategy == "Direct Dataset":
        clean_data = remove_thought_tags(content).strip()

        if file_format == "JSON":
            try:
                parsed = json.loads(clean_data)
                if isinstance(parsed, dict) and "rows" in parsed:
                    clean_data = json.dumps(parsed["rows"], indent=2)
            except json.JSONDecodeError as e:
                return f"Error parsing generated JSON: {e}", None
        elif file_format == "CSV":
            df = safe_parse_csv(clean_data)
            if df.empty:
                return "Error: could not recover any rows from the generated CSV.", None
            clean_data = df.to_csv(index=False)

        target_path = os.path.join(cwd, f"synthetic_dataset.{ext}")
        try:
            with open(target_path, "w", encoding="utf-8") as f:
                f.write(clean_data)
            return f"Dataset saved successfully!\nFile location: {target_path}", target_path
        except OSError as e:
            return f"Error saving dataset: could not write file ({e})", None
    else:
        clean_code = extract_pure_code(content)
        if not clean_code:
            return "No executable Python code found.", None

        before_files = set(os.listdir(cwd))
        try:
            result = run_generated_code(clean_code, cwd)
        except subprocess.TimeoutExpired:
            return f"Execution error: the generated code did not finish within {EXEC_TIMEOUT_SECONDS} seconds and was stopped.", None

        if result.returncode != 0:
            stderr_lines = result.stderr.strip().splitlines()
            last_err = stderr_lines[-1] if stderr_lines else "Unknown error"
            error_kind = "syntax error" if "SyntaxError" in result.stderr else "runtime error"
            return f"The generated code raised a {error_kind}: {last_err}\n\nDetails:\n{result.stderr}", None

        after_files = set(os.listdir(cwd))
        new_files = list(after_files - before_files)

        saved_file_path = None
        if new_files:
            saved_file_path = os.path.join(cwd, sorted(new_files)[0])
        else:
            for e in ["csv", "json", "md", "txt"]:
                candidate = os.path.join(cwd, f"synthetic_dataset.{e}")
                if os.path.exists(candidate):
                    saved_file_path = candidate
                    break

        if saved_file_path and os.path.exists(saved_file_path):
            return f"Code executed successfully!\nFile location: {saved_file_path}", saved_file_path
        else:
            return f"Code executed without error, but no file was written to {cwd}.\nPlease ensure the code includes df.to_csv('synthetic_dataset.{ext}', index=False).", None

## Steps 9 & 10: Build and wire the Gradio UI

Single output window interface that streams either code or dataset, with one-click execution and download.

In [ ]:
def generate_dataset(business_problem, data_type, num_rows, file_format, generation_strategy, model_choice, schema_fields):
    if not business_problem or not business_problem.strip():
        yield "Please enter a business problem."
        return

    num_rows = int(num_rows) if num_rows else 0
    error = validate_inputs(num_rows, file_format)
    if error:
        yield error
        return

    schema_json = build_schema_json(schema_fields)

    if "Gemini" in model_choice:
        provider = generate_with_gemini
    elif "Claude" in model_choice:
        provider = generate_with_claude
    else:
        provider = generate_with_ollama

    if generation_strategy == "Direct Dataset" and num_rows > BATCH_SIZE:
        yield from generate_in_batches(provider, business_problem, data_type, num_rows, file_format, generation_strategy, schema_json)
        return

    generator = provider(business_problem, data_type, num_rows, file_format, generation_strategy, schema_json)
    for full_response in generator:
        yield clean_stream(full_response, generation_strategy)


def update_ui_labels(strategy):
    if strategy == "Direct Dataset":
        return gr.update(label="Generated Dataset"), gr.update(value="Save Dataset to File")
    return gr.update(label="Generated Python Code"), gr.update(value="Run Code Safely")


with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("# Synthetic Data Generator")
    gr.Markdown("Create synthetic datasets for business problems using LLMs.")

    with gr.Row():
        with gr.Column(scale=1):
            business_problem = gr.Textbox(
                label="Business Problem",
                placeholder="e.g. Patient records for a dental clinic",
                lines=3,
            )
            generation_strategy = gr.Radio(
                choices=["Code Generation", "Direct Dataset"],
                value="Code Generation",
                label="Generation Strategy",
            )
            data_type = gr.Dropdown(
                choices=["Tabular", "Text", "Time-series"],
                value="Tabular",
                label="Data Type",
            )
            file_format = gr.Dropdown(
                choices=["CSV", "JSON", "Markdown"],
                value="CSV",
                label="File Format",
            )
            num_rows = gr.Number(
                value=20,
                label="Number of Rows",
                precision=0,
            )
            schema_fields = gr.Textbox(
                label="Schema (optional)",
                placeholder="e.g. name:str, age:int, email:str, signup_date:str",
                lines=1,
            )
            model_choice = gr.Dropdown(
                choices=["Google Gemini", "Anthropic Claude", "Local Ollama (Gemma 4)"],
                value="Google Gemini",
                label="Model Choice",
            )
            generate_btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=2):
            output_display = gr.Code(
                label="Generated Python Code",
                language="python",
                lines=22,
            )
            action_btn = gr.Button("Run Code Safely", variant="secondary")
            status_output = gr.Textbox(
                label="Execution Status",
                interactive=False,
                lines=3,
            )
            file_output = gr.File(
                label="Download Saved File"
            )

    generation_strategy.change(
        fn=update_ui_labels,
        inputs=[generation_strategy],
        outputs=[output_display, action_btn],
    )

    generate_btn.click(
        fn=generate_dataset,
        inputs=[business_problem, data_type, num_rows, file_format, generation_strategy, model_choice, schema_fields],
        outputs=[output_display],
    )

    action_btn.click(
        fn=process_output,
        inputs=[output_display, file_format, generation_strategy],
        outputs=[status_output, file_output],
    )

demo.launch()